In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [1]:
# 计算加入Outler Detector后是否会对Lora模型的表现性能产生显著影响
MODEL_NAME_LIST = ["codebert","plbart", "graphcodebert", "unixcoder",  "codet5","cct5"]
CLUSTER_MODEL_LIST = ["developer_aware","Kmean","project_final"]
N_CLUSTER = 4
BASE_MODEL = "concat"
import os
import scipy.stats as stats
import pandas as pd

In [34]:
import scipy.stats as stats
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def analyze_performance_trend(before, after, alpha=0.05):
    """
    分析性能变化趋势是否显著
    
    Parameters:
    - before: 第一组数据（基准/改进前）
    - after: 第二组数据（改进后）
    - alpha: 显著性水平，默认0.05
    """
    
    print("=" * 60)
    print("性能变化趋势分析")
    print("=" * 60)
    
    # 计算变化量
    differences = after - before
    percent_changes = ((after - before) / before) * 100
    
    print(f"样本数量: {len(before)}")
    print(f"基准组均值: {np.mean(before):.4f}")
    print(f"改进组均值: {np.mean(after):.4f}")
    print(f"平均变化量: {np.mean(differences):.4f}")
    print(f"平均变化百分比: {np.mean(percent_changes):.2f}%")
    
    # 1. 正态性检验（针对差值）
    _, p_norm = stats.shapiro(differences)
    print(f"\n差值正态性检验(Shapiro-Wilk) p值: {p_norm:.4f}")
    
    # 2. 选择合适的检验方法
    if p_norm > alpha:
        # 使用配对t检验
        t_stat, p_value = stats.ttest_rel(after, before)
        test_used = "配对t检验"
        effect_size = np.mean(differences) / np.std(differences, ddof=1)  # Cohen's d
    else:
        # 使用Wilcoxon符号秩检验
        t_stat, p_value = stats.wilcoxon(after, before)
        test_used = "Wilcoxon符号秩检验"
        # 对于非参数检验，使用中位数计算效应量
        effect_size = np.median(differences) / stats.median_abs_deviation(differences)
    
    print(f"\n使用的检验方法: {test_used}")
    print(f"检验统计量: {t_stat:.4f}")
    print(f"p值: {p_value:.4f}")
    print(f"效应量: {effect_size:.4f}")
    
    # 3. 趋势方向判断
    mean_change = np.mean(differences)
    if mean_change > 0:
        direction = "提升"
        trend = "正向"
    else:
        direction = "下降" 
        trend = "负向"
    
    # 4. 结果解释
    if p_value < alpha:
        print(f"\n结论: 性能{dir}是统计显著的 (p < {alpha})")
        print(f"趋势: 显著的{trend}趋势")
    else:
        print(f"\n结论: 性能{dir}不显著 (p ≥ {alpha})")
        print(f"趋势: 无显著趋势")
    
    return t_stat,p_value

In [35]:
import scipy.stats as stats
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def analyze_performance_trend_Pcode(before, after, alpha=0.05, higher_is_better=True):
    """
    分析性能变化趋势是否显著
    
    Parameters:
    - before: 第一组数据（基准/改进前）
    - after: 第二组数据（改进后）
    - alpha: 显著性水平，默认0.05
    - higher_is_better: 指标是否越高越好，默认True
    
    Returns:
    - 'B': 显著更好
    - 'W': 显著变差  
    - 'N': 无显著差异
    """
    
    print("=" * 60)
    print("性能变化趋势分析")
    print("=" * 60)
    
    # 计算变化量
    differences = after - before
    percent_changes = ((after - before) / before) * 100
    
    # print(f"样本数量: {len(before)}")
    # print(f"基准组均值: {np.mean(before):.4f}")
    # print(f"改进组均值: {np.mean(after):.4f}")
    # print(f"平均变化量: {np.mean(differences):.4f}")
    # print(f"平均变化百分比: {np.mean(percent_changes):.2f}%")
    
    # 1. 正态性检验（针对差值）
    _, p_norm = stats.shapiro(differences)
    print(f"\n差值正态性检验(Shapiro-Wilk) p值: {p_norm:.4f}")
    
    # 2. 选择合适的检验方法
    if p_norm > alpha:
        # 使用配对t检验
        t_stat, p_value = stats.ttest_rel(after, before)
        test_used = "配对t检验"
        effect_size = np.mean(differences) / np.std(differences, ddof=1)  # Cohen's d
    else:
        # 使用Wilcoxon符号秩检验
        t_stat, p_value = stats.wilcoxon(after, before)
        test_used = "Wilcoxon符号秩检验"
        # 对于非参数检验，使用中位数计算效应量
        effect_size = np.median(differences) / stats.median_abs_deviation(differences)
    
    print(f"\n使用的检验方法: {test_used}")
    print(f"检验统计量: {t_stat:.4f}")
    print(f"p值: {p_value:.4f}")
    print(f"效应量: {effect_size:.4f}")
    
    # 3. 趋势方向判断
    mean_change = np.mean(differences)
    if mean_change > 0:
        direction = "提升"
        trend = "正向"
    else:
        direction = "下降" 
        trend = "负向"
    
    # 4. 确定返回结果
    result_code = 'N'  # 默认无显著差异
    
    if p_value < alpha:
        # 有显著差异
        if higher_is_better:
            # 指标越高越好
            if mean_change > 0:
                result_code = 'B'  # 显著更好
            else:
                result_code = 'W'  # 显著变差
        else:
            # 指标越低越好（如响应时间）
            if mean_change < 0:
                result_code = 'B'  # 显著更好
            else:
                result_code = 'W'  # 显著变差
        
        # print(f"\n结论: 性能{direction}是统计显著的 (p < {alpha})")
        # print(f"趋势: 显著的{trend}趋势")
        # print(f"返回代码: {result_code}")
    else:
        # 无显著差异
        result_code = 'N'
        # print(f"\n结论: 性能{direction}不显著 (p ≥ {alpha})")
        # print(f"趋势: 无显著趋势")
        # print(f"返回代码: {result_code}")
    
    return result_code

In [ ]:

### Base To Lora significance
for metric in ["f1","gmean"]:
    significance_table=[]
    for cluster_model in CLUSTER_MODEL_LIST:
        line=[]
        for model_name in MODEL_NAME_LIST:
            base_data=[]
            lora_data=[]
            for result_time in ["result0", "result1", "result2", "result3", "result4"]:
                lora_path = os.path.join(f"result/{cluster_model}/{str(N_CLUSTER)}/{result_time}/{model_name}", "lora_results.csv")
                base_path = os.path.join(f"result/{cluster_model}/{str(N_CLUSTER)}/{result_time}/{model_name}", "base_results.csv")

                lora_data.append(pd.read_csv(lora_path,index_col=0).loc["all"][metric])
                base_data.append(pd.read_csv(base_path,index_col=0).loc["all"][metric])

            # t_stat,p_value=stats.ttest_rel(base_data,lora_data)
            Pcode=analyze_performance_trend_Pcode(np.array(base_data),np.array(lora_data))
            line.append(Pcode)
        significance_table.append(line)
    print(f"for {metric} table is {significance_table}")
    pd.DataFrame(significance_table,index=CLUSTER_MODEL_LIST,columns=MODEL_NAME_LIST).to_csv(f"result/Significance/base_to_lora_significance_compare_{metric}.csv")
##这块结果显示加入Outlier Detector之后的差异是不显著的

性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.0277

使用的检验方法: Wilcoxon符号秩检验
检验统计量: 0.0000
p值: 0.0625
效应量: 39.3233
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.3318

使用的检验方法: 配对t检验
检验统计量: 11.1661
p值: 0.0004
效应量: 4.9936
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.0495

使用的检验方法: Wilcoxon符号秩检验
检验统计量: 0.0000
p值: 0.0625
效应量: 35.8567
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.7049

使用的检验方法: 配对t检验
检验统计量: 17.4965
p值: 0.0001
效应量: 7.8247
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.3748

使用的检验方法: 配对t检验
检验统计量: 4.2839
p值: 0.0128
效应量: 1.9158
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.6735

使用的检验方法: 配对t检验
检验统计量: 3.6721
p值: 0.0214
效应量: 1.6422
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.9041

使用的检验方法: 配对t检验
检验统计量: 8.2890
p值: 0.0012
效应量: 3.7070
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.4669

使用的检验方法: 配对t检验
检验统计量: 3.3332
p值: 0.0290
效应量: 1.4907
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.7860

使用的检验方法: 配对t检验
检验统计量: 7.5921
p值: 0.0016
效应量: 3.3953
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.9970

使用的检验方法: 配对t检验
检验统计量: 5.8972
p值: 0.0041
效应量: 2.6373
性能变化趋势分析

差值正态性检验(Sh

In [ ]:
### Base To Lora Pvalue
for metric in ["f1","gmean"]:
    significance_table=[]
    for cluster_model in CLUSTER_MODEL_LIST:
        line=[]
        for model_name in MODEL_NAME_LIST:
            base_data=[]
            lora_data=[]
            for result_time in ["result0", "result1", "result2", "result3", "result4"]:
                lora_path = os.path.join(f"result/{cluster_model}/{str(N_CLUSTER)}/{result_time}/{model_name}", "lora_results.csv")
                base_path = os.path.join(f"result/{cluster_model}/{str(N_CLUSTER)}/{result_time}/{model_name}", "base_results.csv")

                lora_data.append(pd.read_csv(lora_path,index_col=0).loc["all"][metric])
                base_data.append(pd.read_csv(base_path,index_col=0).loc["all"][metric])

            # t_stat,p_value=stats.ttest_rel(base_data,lora_data)
            t,p_value=analyze_performance_trend(np.array(base_data),np.array(lora_data))
            line.append(round(p_value,4))
        significance_table.append(line)
    print(f"for {metric} table is {significance_table}")
    pd.DataFrame(significance_table,index=CLUSTER_MODEL_LIST,columns=MODEL_NAME_LIST).to_csv(f"result/Significance/base_to_lora_pvalue_compare_{metric}.csv")
##这块结果显示加入Outlier Detector之后的差异是不显著的

性能变化趋势分析
样本数量: 5
基准组均值: 0.4227
改进组均值: 0.4486
平均变化量: 0.0259
平均变化百分比: 6.14%

差值正态性检验(Shapiro-Wilk) p值: 0.0277

使用的检验方法: Wilcoxon符号秩检验
检验统计量: 0.0000
p值: 0.0625
效应量: 39.3233

结论: 性能<built-in function dir>不显著 (p ≥ 0.05)
趋势: 无显著趋势
性能变化趋势分析
样本数量: 5
基准组均值: 0.4715
改进组均值: 0.4800
平均变化量: 0.0085
平均变化百分比: 1.79%

差值正态性检验(Shapiro-Wilk) p值: 0.3318

使用的检验方法: 配对t检验
检验统计量: 11.1661
p值: 0.0004
效应量: 4.9936

结论: 性能<built-in function dir>是统计显著的 (p < 0.05)
趋势: 显著的正向趋势
性能变化趋势分析
样本数量: 5
基准组均值: 0.4532
改进组均值: 0.4725
平均变化量: 0.0192
平均变化百分比: 4.25%

差值正态性检验(Shapiro-Wilk) p值: 0.0495

使用的检验方法: Wilcoxon符号秩检验
检验统计量: 0.0000
p值: 0.0625
效应量: 35.8567

结论: 性能<built-in function dir>不显著 (p ≥ 0.05)
趋势: 无显著趋势
性能变化趋势分析
样本数量: 5
基准组均值: 0.4549
改进组均值: 0.4725
平均变化量: 0.0177
平均变化百分比: 3.88%

差值正态性检验(Shapiro-Wilk) p值: 0.7049

使用的检验方法: 配对t检验
检验统计量: 17.4965
p值: 0.0001
效应量: 7.8247

结论: 性能<built-in function dir>是统计显著的 (p < 0.05)
趋势: 显著的正向趋势
性能变化趋势分析
样本数量: 5
基准组均值: 0.4621
改进组均值: 0.4707
平均变化量: 0.0086
平均变化百分比: 1.86%

差值正态性检验(Shapiro-Wilk) p值: 0.374

In [ ]:

### Lora To outlier pvalue
for metric in ["f1","gmean"]:
    significance_table=[]
    for cluster_model in CLUSTER_MODEL_LIST:
        line=[]
        for model_name in MODEL_NAME_LIST:
            lora_data=[]
            outlier_data=[]
            for result_time in ["result0", "result1", "result2", "result3", "result4"]:
                lora_path = os.path.join(f"result/{cluster_model}/{str(N_CLUSTER)}/{result_time}/{model_name}", "lora_results.csv")
                outlier_path = os.path.join(f"result/{cluster_model}/{str(N_CLUSTER)}/{result_time}/{model_name}", "outlier_results.csv")

                lora_data.append(pd.read_csv(lora_path,index_col=0).loc["all"][metric].item())
                outlier_data.append(pd.read_csv(outlier_path,index_col=0).loc["all"][metric].item())

            # t_stat,p_value=stats.ttest_rel(lora_data,outlier_data)
            t_stat,p_value=analyze_performance_trend(np.array(lora_data),np.array(outlier_data))
            line.append(round(float(p_value),4))
        significance_table.append(line)
    print(f"for {metric} table is {significance_table}")
    pd.DataFrame(significance_table,index=CLUSTER_MODEL_LIST,columns=MODEL_NAME_LIST).to_csv(f"result/Significance/lora_to_outlier_pvalue_compare_{metric}.csv")
##这块结果显示加入Outlier Detector之后的差异是不显著的

性能变化趋势分析
样本数量: 5
基准组均值: 0.4486
改进组均值: 0.4430
平均变化量: -0.0056
平均变化百分比: -1.24%

差值正态性检验(Shapiro-Wilk) p值: 0.2186

使用的检验方法: 配对t检验
检验统计量: -4.7516
p值: 0.0090
效应量: -2.1250

结论: 性能<built-in function dir>是统计显著的 (p < 0.05)
趋势: 显著的负向趋势
性能变化趋势分析
样本数量: 5
基准组均值: 0.4800
改进组均值: 0.4799
平均变化量: -0.0001
平均变化百分比: -0.02%

差值正态性检验(Shapiro-Wilk) p值: 0.5115

使用的检验方法: 配对t检验
检验统计量: -0.1360
p值: 0.8984
效应量: -0.0608

结论: 性能<built-in function dir>不显著 (p ≥ 0.05)
趋势: 无显著趋势
性能变化趋势分析
样本数量: 5
基准组均值: 0.4725
改进组均值: 0.4648
平均变化量: -0.0077
平均变化百分比: -1.63%

差值正态性检验(Shapiro-Wilk) p值: 0.1273

使用的检验方法: 配对t检验
检验统计量: -3.0590
p值: 0.0377
效应量: -1.3680

结论: 性能<built-in function dir>是统计显著的 (p < 0.05)
趋势: 显著的负向趋势
性能变化趋势分析
样本数量: 5
基准组均值: 0.4725
改进组均值: 0.4664
平均变化量: -0.0061
平均变化百分比: -1.30%

差值正态性检验(Shapiro-Wilk) p值: 0.7451

使用的检验方法: 配对t检验
检验统计量: -4.9513
p值: 0.0078
效应量: -2.2143

结论: 性能<built-in function dir>是统计显著的 (p < 0.05)
趋势: 显著的负向趋势
性能变化趋势分析
样本数量: 5
基准组均值: 0.4707
改进组均值: 0.4682
平均变化量: -0.0025
平均变化百分比: -0.53%

差值正态性检验(Shapiro-Wilk) p值: 0.

In [ ]:

### Lora To outlier Significance
for metric in ["f1","gmean"]:
    significance_table=[]
    for cluster_model in CLUSTER_MODEL_LIST:
        line=[]
        for model_name in MODEL_NAME_LIST:
            lora_data=[]
            outlier_data=[]
            for result_time in ["result0", "result1", "result2", "result3", "result4"]:
                lora_path = os.path.join(f"result/{cluster_model}/{str(N_CLUSTER)}/{result_time}/{model_name}", "lora_results.csv")
                outlier_path = os.path.join(f"result/{cluster_model}/{str(N_CLUSTER)}/{result_time}/{model_name}", "outlier_results.csv")

                lora_data.append(pd.read_csv(lora_path,index_col=0).loc["all"][metric].item())
                outlier_data.append(pd.read_csv(outlier_path,index_col=0).loc["all"][metric].item())

            # t_stat,p_value=stats.ttest_rel(lora_data,outlier_data)
            Pcode=analyze_performance_trend_Pcode(np.array(lora_data),np.array(outlier_data))
            line.append(Pcode)
        significance_table.append(line)
    print(f"for {metric} table is {significance_table}")
    pd.DataFrame(significance_table,index=CLUSTER_MODEL_LIST,columns=MODEL_NAME_LIST).to_csv(f"result/Significance/lora_to_outlier_significance_compare_{metric}.csv")
##这块结果显示加入Outlier Detector之后的差异是不显著的

性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.2186

使用的检验方法: 配对t检验
检验统计量: -4.7516
p值: 0.0090
效应量: -2.1250
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.5115

使用的检验方法: 配对t检验
检验统计量: -0.1360
p值: 0.8984
效应量: -0.0608
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.1273

使用的检验方法: 配对t检验
检验统计量: -3.0590
p值: 0.0377
效应量: -1.3680
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.7451

使用的检验方法: 配对t检验
检验统计量: -4.9513
p值: 0.0078
效应量: -2.2143
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.6336

使用的检验方法: 配对t检验
检验统计量: -2.2628
p值: 0.0864
效应量: -1.0119
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.5276

使用的检验方法: 配对t检验
检验统计量: 1.0876
p值: 0.3379
效应量: 0.4864
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.7183

使用的检验方法: 配对t检验
检验统计量: 1.2068
p值: 0.2940
效应量: 0.5397
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.8237

使用的检验方法: 配对t检验
检验统计量: 18.0150
p值: 0.0001
效应量: 8.0565
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.1165

使用的检验方法: 配对t检验
检验统计量: 3.9160
p值: 0.0173
效应量: 1.7513
性能变化趋势分析

差值正态性检验(Shapiro-Wilk) p值: 0.1025

使用的检验方法: 配对t检验
检验统计量: 6.5673
p值: 0.0028
效应量: 2.9370
性能变化趋势分析

差值正态性检验(Shapiro-Wil